# OncologyBreastScanner — Retraining on CBIS-DDSM

Run all cells in order. A GPU runtime is **required** (Runtime → Change runtime type → T4 GPU).

**Before starting:** upload your `kaggle.json` when prompted, or use the Kaggle Hub method (no file needed).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/breast_cancer_model'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working in {WORK_DIR}')

## Step 1: Install dependencies

In [ ]:
!pip install -q kagglehub tensorflow opencv-python scikit-learn matplotlib seaborn pandas

## Step 2: Download CBIS-DDSM from Kaggle

In [ ]:
import kagglehub
print('Downloading CBIS-DDSM from Kaggle (may take 10-15 min)...')
path = kagglehub.dataset_download('awsaf49/cbis-ddsm-breast-cancer-image-dataset')
print(f'Downloaded to: {path}')
!ls -la "$path"

## Step 3: Prepare dataset (label mapping + image copy)

In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

KAGGLE_DIR = Path(path)
OUTPUT_DIR = Path('/content/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = OUTPUT_DIR / 'images'
IMAGES_DIR.mkdir(exist_ok=True)

CLASS_LABELS = [
    'Density1Benign', 'Density1Malignant',
    'Density2Benign', 'Density2Malignant',
    'Density3Benign', 'Density3Malignant',
    'Density4Benign', 'Density4Malignant',
]

def pathology_to_binary(p):
    p = str(p).strip().upper()
    if 'BENIGN' in p: return 'Benign'
    elif 'MALIGNANT' in p: return 'Malignant'
    return None

def get_density_col(row):
    for col in ['breast density', 'breast_density', 'density']:
        val = row.get(col)
        if val is not None and not pd.isna(val):
            return val
    return None

def extract_uid(dcm_path):
    parts = str(dcm_path).strip().split('/')
    return parts[-2] if len(parts) >= 2 else None

def find_jpeg_for_dicom(dcm_path, jpeg_dir):
    uid = extract_uid(dcm_path)
    if not uid: return None
    uid_dir = os.path.join(jpeg_dir, uid)
    if not os.path.isdir(uid_dir): return None
    jpgs = sorted(f for f in os.listdir(uid_dir) if f.lower().endswith(('.jpg','.jpeg','.png')))
    return os.path.join(uid_dir, jpgs[0]) if jpgs else None

csv_dir = KAGGLE_DIR / 'csv'
jpeg_dir = KAGGLE_DIR / 'jpeg'

records = []
for csv_name in [
    'mass_case_description_train_set.csv',
    'mass_case_description_test_set.csv',
    'calc_case_description_train_set.csv',
    'calc_case_description_test_set.csv',
]:
    csv_path = csv_dir / csv_name
    if not csv_path.exists(): continue
    df = pd.read_csv(csv_path)
    for _, row in df.iterrows():
        density = get_density_col(row)
        pathology = row.get('pathology')
        try:
            d = int(float(str(density).strip()))
            p = pathology_to_binary(pathology)
            if d is None or p is None or d < 1 or d > 4: continue
            label = f'Density{d}{p}'
            if label not in CLASS_LABELS: continue
        except: continue

        jpeg_path = None
        for col in ['cropped image file path', 'image file path']:
            dcm = row.get(col)
            if not dcm or pd.isna(dcm): continue
            jpeg_path = find_jpeg_for_dicom(dcm, jpeg_dir)
            if jpeg_path: break

        if jpeg_path:
            records.append({'source': jpeg_path, 'label': label})

print(f'Total usable images: {len(records)}')
for cls in CLASS_LABELS:
    c = sum(1 for r in records if r['label'] == cls)
    if c > 0: print(f'  {cls:25s} {c}')

# Copy images (resized to 224x224)
for i, rec in enumerate(records):
    dst = IMAGES_DIR / f"{rec['label']}_{i:05d}.png"
    if dst.exists(): continue
    img = cv2.imread(rec['source'])
    if img is None: continue
    img = cv2.resize(img, (224, 224))
    cv2.imwrite(str(dst), img)

# Split into train/val
all_files = sorted(os.listdir(IMAGES_DIR))
all_labels = [f.rsplit('_', 1)[0] for f in all_files]
train_files, val_files = train_test_split(
    all_files, test_size=0.15, stratify=all_labels, random_state=42
)

train_df = pd.DataFrame({'filename': train_files,
    'label': [f.rsplit('_',1)[0] for f in train_files]})
val_df = pd.DataFrame({'filename': val_files,
    'label': [f.rsplit('_',1)[0] for f in val_files]})
train_df.to_csv(OUTPUT_DIR / 'train_set.csv', index=False)
val_df.to_csv(OUTPUT_DIR / 'val_set.csv', index=False)

print(f'\nTrain: {len(train_df)}, Val: {len(val_df)}')
print(f'Images saved to {IMAGES_DIR}')

## Step 4: Build model

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, GlobalAveragePooling2D, GlobalMaxPooling2D, Concatenate, Input
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1_l2

def build_model(input_shape=(224,224,3), n_classes=8):
    conv_base = DenseNet201(weights='imagenet', include_top=False,
                            input_shape=input_shape, pooling=None)
    conv_base.trainable = False
    inputs = Input(shape=input_shape)
    x = conv_base(inputs)
    x = Concatenate()([GlobalAveragePooling2D()(x), GlobalMaxPooling2D()(x)])
    x = BatchNormalization()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=l1_l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation='relu', kernel_regularizer=l1_l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x)
    outputs = Dense(n_classes, activation='softmax')(x)
    return Model(inputs=inputs, outputs=outputs), conv_base

print('Building model...')
model, conv_base = build_model()
model.summary()

## Step 5: Train

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_df = pd.read_csv(OUTPUT_DIR / 'train_set.csv')
val_df = pd.read_csv(OUTPUT_DIR / 'val_set.csv')

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    brightness_range=(0.9, 1.1),
)
val_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    train_df, directory=str(IMAGES_DIR), x_col='filename', y_col='label',
    target_size=(224,224), batch_size=32, class_mode='categorical',
    shuffle=True, seed=42
)
val_flow = val_gen.flow_from_dataframe(
    val_df, directory=str(IMAGES_DIR), x_col='filename', y_col='label',
    target_size=(224,224), batch_size=32, class_mode='categorical',
    shuffle=False
)

class_weights = compute_class_weight(
    'balanced',
    classes=np.array(list(train_flow.class_indices.keys())),
    y=train_df['label'].values
)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

In [ ]:
print('Phase 1: Training head only...')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history1 = model.fit(
    train_flow, validation_data=val_flow, epochs=20,
    class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint('phase1_best.weights.h5',
            monitor='val_accuracy', save_best_only=True,
            save_weights_only=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
            factor=0.5, patience=3, min_lr=1e-6),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss',
            patience=6, restore_best_weights=True),
    ],
    verbose=1
)

In [ ]:
print('Phase 2: Fine-tuning with unfrozen layers...')
for layer in conv_base.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history2 = model.fit(
    train_flow, validation_data=val_flow, epochs=60,
    initial_epoch=20, class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint('phase2_best.weights.h5',
            monitor='val_accuracy', save_best_only=True,
            save_weights_only=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
            factor=0.5, patience=4, min_lr=1e-7),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss',
            patience=8, restore_best_weights=True),
    ],
    verbose=1
)

## Step 6: Save to Google Drive & download

In [ ]:
import time
timestamp = time.strftime('%Y%m%d_%H%M%S')

model.save(f'breast_model_{timestamp}.h5')
model.save_weights(f'breast_weights_{timestamp}.h5')
print(f'Saved: breast_model_{timestamp}.h5')
print(f'Saved: breast_weights_{timestamp}.h5')

# Copy to Drive for persistence
!cp breast_model_{timestamp}.h5 "$WORK_DIR/"
!cp breast_weights_{timestamp}.h5 "$WORK_DIR/"
print(f'\nCopied to Google Drive: {WORK_DIR}/')

# Also download via browser
from google.colab import files
files.download(f'breast_weights_{timestamp}.h5')

## Step 7: Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

val_flow.reset()
y_prob = model.predict(val_flow, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = val_flow.classes
class_names = list(val_flow.class_indices.keys())

print(f'\nAccuracy: {np.mean(y_pred == y_true):.4f}'
      f' ({(np.mean(y_pred == y_true)*100):.2f}%)')
print(f'Top-2 Accuracy: {np.mean(np.any("
    [np.argsort(p)[-2:] == t for p,t in zip(y_prob, y_true)], axis=1)):.4f}')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names,
                            zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c.replace('Density','D') for c in class_names],
            yticklabels=[c.replace('Density','D') for c in class_names])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
files.download('confusion_matrix.png')

In [ ]:
import matplotlib.pyplot as plt
full_history = history1.history
for k in history2.history:
    if k in full_history:
        full_history[k] = full_history[k] + history2.history[k]
    else:
        full_history[k] = history2.history[k]

plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(full_history['loss'], label='Train Loss')
plt.plot(full_history['val_loss'], label='Val Loss')
plt.axvline(x=20, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Epoch')
plt.legend()
plt.title('Loss')
plt.subplot(1,2,2)
plt.plot(full_history['accuracy'], label='Train Acc')
plt.plot(full_history['val_accuracy'], label='Val Acc')
plt.axvline(x=20, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Epoch')
plt.legend()
plt.title('Accuracy')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Done! Download the weights from Google Drive or use files.download above.')